# ArmyOfSafeguards — Colab GPU SFT (experts)

This notebook fine-tunes one expert model on **native curriculum JSONL** and logs metrics to `training/experts/sft_metrics.jsonl`.

**Prereqs**
- Colab runtime: **GPU**
- Optional: Hugging Face token for gated datasets (`HF_TOKEN`)

In [1]:
# (Colab) Check GPU
!nvidia-smi

import sys, torch
print('python', sys.version)
print('torch', torch.__version__)
print('cuda available', torch.cuda.is_available())

Fri Apr 10 06:35:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# (Colab) Clone repo + install deps
# If you're running from your own fork, change the URL.

REPO_URL = "https://github.com/faketut/ArmyOfSafeguards.git"
REPO_DIR = "ArmyOfSafeguards"

!rm -rf "$REPO_DIR"
!git clone "$REPO_URL" "$REPO_DIR"

%cd $REPO_DIR

!pip -q install -r requirements.txt
# Optional (only needed if you pass --lora):
!pip -q install peft

Cloning into 'ArmyOfSafeguards'...
remote: Enumerating objects: 669, done.
remote: Counting objects: 100% (169/169), done.
remote: Compressing objects: 100% (96/96), done.
remote: Total 669 (delta 80), reused 134 (delta 64), pack-reused 500 (from 1)
Receiving objects: 100% (669/669), 14.95 MiB | 41.60 MiB/s, done.
Resolving deltas: 100% (317/317), done.
/content/ArmyOfSafeguards
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 106.0 MB/s eta 0:00:00


In [ ]:
# Re-run the login cell after installing huggingface_hub
import os
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

if hf_token:
    # Set HF_TOKEN as an environment variable for the CLI to pick up
    os.environ['HF_TOKEN'] = hf_token
    !hf auth login
else:
    print("HF_TOKEN not set. If you hit gated dataset errors, set HF_TOKEN and rerun.")

In [ ]:
# Build native curriculum SFT JSONL (choose one expert)
# Options for EXPERT: jailbreak | toxicity | sexual | factuality

EXPERT = "sexual"
OUT_JSONL = f"/content/{EXPERT}_native.jsonl"  # put under /content for easy download

# NOTE: use {VAR} interpolation (Python -> shell) instead of $VAR (shell env var).
!python training/experts/build_expert_sft_jsonl.py --expert "{EXPERT}" --out "{OUT_JSONL}"

import os
print("jsonl", OUT_JSONL)
print("exists", os.path.exists(OUT_JSONL))
print("rows", sum(1 for _ in open(OUT_JSONL, "r", encoding="utf-8")))

In [5]:
# Run SFT (writes metrics to a JSONL registry)
# Tip: use --fp16 on T4/A100; use --bf16 on A100.

OUTPUT_DIR = f"experts/artifacts/{EXPERT}_ft"
METRICS_REG = f"/content/sft_metrics_{EXPERT}.jsonl"  # explicit absolute path

# NOTE: use {VAR} interpolation (Python -> shell) instead of $VAR (shell env var).
!python training/common/sequence_classifier_train.py \
  --data "{OUT_JSONL}" \
  --domain "{EXPERT}" \
  --output-dir "{OUTPUT_DIR}" \
  --metrics-registry "{METRICS_REG}" \
  --epochs 2 \
  --batch 16 \
  --grad-accum 1 \
  --lr 2e-5 \
  --max-length 256 \
  --bf16

print("Saved model to", OUTPUT_DIR)
print("Metrics registry:", METRICS_REG)
print("Runtime env var:")
print({
    "toxicity": "AOS_TOXICITY_MODEL",
    "sexual": "AOS_SEXUAL_MODEL",
    "factuality": "AOS_FACTUALITY_MODEL",
    "jailbreak": "AOS_JAILBREAK_MODEL",
}.get(EXPERT, "AOS_<EXPERT>_MODEL"), "=", OUTPUT_DIR)

Saved model to experts/artifacts/sexual_ft
Runtime env var:
AOS_SEXUAL_MODEL = experts/artifacts/sexual_ft


In [ ]:
# View / compare runs
# NOTE: the metrics registry is resolved relative to the repo root *as imported*.
import os
from pathlib import Path
import training.common.sequence_classifier_train as sft

print("cwd", os.getcwd())
print("repo_root", sft._REPO_ROOT)
reg = Path(sft.DEFAULT_SFT_METRICS_REGISTRY)
reg_abs = reg if reg.is_absolute() else (Path(sft._REPO_ROOT) / reg)
print("registry", reg_abs)

!ls -lh "{reg_abs}" || true
!python training/experts/summarize_sft_metrics.py "{reg_abs}"